<a href="https://colab.research.google.com/github/chamindu002/Research/blob/main/pure_vector_search_name_matching.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## CELL 1 - Install & Imports

In [ ]:
# =============================================================================
# CELL 1 - Install & Imports
# =============================================================================
# !pip install Unidecode faiss-cpu rapidfuzz scikit-learn pandas numpy -q

import pandas as pd
import numpy as np
import re
import time
import pickle
import os
import sys
from unidecode import unidecode
from typing import List, Tuple, Dict, Set
from collections import Counter

import faiss
from sklearn.feature_extraction.text import TfidfVectorizer
from rapidfuzz import fuzz
from rapidfuzz.distance import Levenshtein, Jaro

print("✓ All imports done! (NO pre-trained models loaded)")

✓ All imports done! (NO pre-trained models loaded)


## CELL 2 - Mount Drive

In [ ]:
# =============================================================================
# CELL 2 - Mount Drive
# =============================================================================
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted.


## CELL 3 - Load Data

In [ ]:
# =============================================================================
# CELL 3 - Load Data
# =============================================================================
print("Loading 100,000 PEP records...")

df = pd.read_csv(
    "/content/drive/My Drive/Research/Data/pep_full_list.csv",
    nrows=100000,
    dtype=str,
    low_memory=False,
    encoding="utf-8",
    on_bad_lines='skip'
)

df["category"] = "pep"
if "identifiers" in df.columns:
    df = df.rename(columns={"identifiers": "id"})

df = df[["name", "aliases", "birth_date", "addresses", "id", "category"]].copy()
df = df.loc[:, ~df.columns.duplicated()]
df = df.replace(["NaN", "null", "NULL", "nan"], [None, None, None, None])

print(f"Loaded {len(df):,} rows → Ready!")
df.head(3)


Loading 100,000 PEP records...
Loaded 100,000 rows → Ready!


,name,aliases,birth_date,addresses,id,category
0,"AW Kim Geok, PBS, PPA(P)",NaN,NaN,NaN,NK-24KdLmG7SRFdS5GgJbgLSs,pep
1,Christophane FOO,NaN,NaN,NaN,NK-24W9aosPLEsueQ5grGVNiK,pep
2,DEMIDOVICH VASILIJ,Демидович Василий Николаевич,NaN,NaN,NK-24isa6tapAYCqtEnqbUdzv,pep


## CELL 4 - Text Normalization (Same as yours)

In [ ]:
# =============================================================================
# CELL 4 - Text Normalization (UPDATED - better apostrophe & concat handling)
# =============================================================================
def normalize_and_transliterate(text: str) -> str:
    if pd.isna(text) or not text:
        return ""
    text = str(text)
    text = unidecode(text)
    text = text.lower()

    # Insert space before uppercase-to-lowercase transitions (CamelCase splitting)
    # "VladmirPutin" → "Vladmir Putin" before lowering
    original = unidecode(str(text)).strip()
    # Re-do on original case to detect camel
    raw = unidecode(str(pd.isna(text) and "" or text))
    # Actually let's work from the original input again
    pass  # we handle this below

    text = re.sub(r"[^a-z0-9\s.]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_and_transliterate(text: str) -> str:
    """Enhanced normalization with CamelCase splitting and apostrophe handling."""
    if pd.isna(text) or not text:
        return ""
    text = str(text)
    text = unidecode(text)

    # CamelCase split: "VladmirPutin" → "Vladmir Putin"
    text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text)

    text = text.lower()

    # Apostrophe → space: "o'connor" → "o connor" (keep as two tokens)
    text = text.replace("'", " ")

    text = re.sub(r"[^a-z0-9\s.]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


# Re-apply cleaning with the new normalizer
df["name_clean"] = df["name"].apply(normalize_and_transliterate)

def split_aliases(alias_str):
    if pd.isna(alias_str):
        return []
    return [normalize_and_transliterate(a.strip()) for a in str(alias_str).split(";")]

df["aliases_clean"] = df["aliases"].apply(split_aliases)
df = df[df["name_clean"].str.len() > 0].reset_index(drop=True)

print(f"Names cleaned! {len(df):,} rows ready.")

# Verify the fix
print("\nNormalization checks:")
for t in ["VladmirPutin", "O'Connor", "V. Putin", "CocaCola", "Britney Spears"]:
    print(f"  '{t}' → '{normalize_and_transliterate(t)}'")


# =============================================================================
# ⚠️  IMPORTANT: After running this Cell 4, you MUST re-run Cell 5 and Cell 6
# ⚠️  to rebuild the feature extractor and FAISS index with the new normalization.
# =============================================================================


Names cleaned! 100,000 rows ready.

Normalization checks:
  'VladmirPutin' → 'vladmir putin'
  'O'Connor' → 'o connor'
  'V. Putin' → 'v. putin'
  'CocaCola' → 'coca cola'
  'Britney Spears' → 'britney spears'


In [ ]:
# =============================================================================
# CELL 5 - Enhanced Feature Extractor
# =============================================================================
class EnhancedFeatureExtractor:
    """
    Improved feature extraction with better initial handling,
    phonetic features, and optimized vectorization.
    """

    def __init__(self):
        # Wider char n-gram range to catch more typo patterns
        self.char_vectorizer = TfidfVectorizer(
            analyzer='char_wb',          # char_wb = word-boundary aware
            ngram_range=(2, 5),           # wider range catches more patterns
            max_features=8000,
            lowercase=True,
            sublinear_tf=True            # dampens high-frequency n-grams
        )
        self.word_vectorizer = TfidfVectorizer(
            analyzer='word',
            ngram_range=(1, 2),
            max_features=5000,
            lowercase=True,
            sublinear_tf=True
        )
        self.fitted = False

    def expand_initials(self, name: str) -> List[str]:
        """Generate variants: remove dots, collapse spaces."""
        variants = [name]
        # "v. putin" → "v putin"
        no_dots = re.sub(r'([a-z])\s*\.\s*', r'\1 ', name).strip()
        if no_dots != name:
            variants.append(no_dots)
        # Also stripped version
        no_dots_compact = re.sub(r'\.\s*', '', name).strip()
        if no_dots_compact not in variants:
            variants.append(no_dots_compact)
        return variants

    def get_consonant_skeleton(self, name: str) -> str:
        return re.sub(r'[aeiou\s.]', '', name.lower())

    def fit(self, names: List[str]):
        print("Building feature vocabularies...")
        expanded = []
        for n in names:
            expanded.extend(self.expand_initials(n))
        self.char_vectorizer.fit(expanded)
        self.word_vectorizer.fit(expanded)
        self.fitted = True
        print(f"  ✓ Char features: {len(self.char_vectorizer.get_feature_names_out())}")
        print(f"  ✓ Word features: {len(self.word_vectorizer.get_feature_names_out())}")

    def transform(self, name: str) -> np.ndarray:
        variants = self.expand_initials(name)
        char_vecs = self.char_vectorizer.transform(variants).toarray()
        word_vecs = self.word_vectorizer.transform(variants).toarray()
        char_vec = char_vecs.mean(axis=0)
        word_vec = word_vecs.mean(axis=0)
        return np.concatenate([char_vec, word_vec]).astype('float32')

    def transform_batch(self, names: List[str], batch_size: int = 5000) -> np.ndarray:
        """Batch transform with progress reporting."""
        total = len(names)
        results = []
        for i in range(0, total, batch_size):
            batch = names[i:i+batch_size]
            batch_vecs = np.array([self.transform(n) for n in batch])
            results.append(batch_vecs)
            done = min(i + batch_size, total)
            print(f"  Transformed {done:,}/{total:,} ({100*done/total:.0f}%)", end='\r')
        print()
        return np.vstack(results)

print("✓ EnhancedFeatureExtractor defined")

✓ EnhancedFeatureExtractor defined


In [ ]:
# =============================================================================
# CELL 6 - Build Vectors & FAISS Index
# =============================================================================
feature_extractor = EnhancedFeatureExtractor()
all_names = df["name_clean"].tolist()

print(f"Fitting on {len(all_names):,} names...")
feature_extractor.fit(all_names)

print("\nTransforming all names to vectors...")
all_vectors = feature_extractor.transform_batch(all_names)
print(f"✓ Vectors: {all_vectors.shape}")

# Normalize & build FAISS index
faiss.normalize_L2(all_vectors)
dimension = all_vectors.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(all_vectors)
print(f"✓ FAISS index: {index.ntotal:,} vectors, {dimension} dims")


Fitting on 100,000 names...
Building feature vocabularies...
  ✓ Char features: 8000
  ✓ Word features: 5000

Transforming all names to vectors...
  Transformed 100,000/100,000 (100%)
✓ Vectors: (100000, 13000)
✓ FAISS index: 100,000 vectors, 13000 dims


## CELL 7 - Build FAISS Index (Fast Similarity Search)

In [ ]:
# =============================================================================
# CELL 7 - Enhanced Composite Similarity (FIXED - no penalty on initials)
# =============================================================================
def _detect_initials(name: str) -> Tuple[List[str], List[str]]:
    """
    Split name into initials and full words.
    'v. putin'  → initials=['v'], words=['putin']
    'j. f. kennedy' → initials=['j','f'], words=['kennedy']
    'vladimir putin' → initials=[], words=['vladimir','putin']
    """
    tokens = name.replace('.', ' . ').split()
    initials, words = [], []
    i = 0
    while i < len(tokens):
        tok = tokens[i]
        if len(tok) == 1 and tok.isalpha():
            initials.append(tok)
            if i + 1 < len(tokens) and tokens[i+1] == '.':
                i += 1
        elif tok != '.':
            words.append(tok)
        i += 1
    return initials, words


def _has_any_initial(name1: str, name2: str) -> bool:
    """Check if EITHER name contains an initial (single letter token)."""
    init1, _ = _detect_initials(name1)
    init2, _ = _detect_initials(name2)
    return bool(init1) or bool(init2)


def _initial_aware_score(name1: str, name2: str) -> float:
    """
    Handles initial-to-fullname matching.
    'v. putin' vs 'vladimir putin' → high score
    'v. putin' vs 'roman putin'   → lower
    Returns -1.0 if neither name has initials.
    """
    init1, words1 = _detect_initials(name1)
    init2, words2 = _detect_initials(name2)

    if not init1 and not init2:
        return -1.0

    if len(init1) >= len(init2):
        initials_side, words_init_side = init1, words1
        initials_other, words_other = init2, words2
    else:
        initials_side, words_init_side = init2, words2
        initials_other, words_other = init1, words1

    # Match shared full words (e.g. "putin" in both)
    shared_words = set(w.lower() for w in words_init_side) & set(w.lower() for w in words_other)
    remaining_other = [w for w in words_other if w.lower() not in shared_words]

    # Match initials against remaining full words by first letter
    matched_initials = 0
    used = set()
    for ini in initials_side:
        for j, w in enumerate(remaining_other):
            if j not in used and len(w) > 0 and w[0] == ini:
                matched_initials += 1
                used.add(j)
                break

    total_parts = len(initials_side) + len(words_init_side)
    if total_parts == 0:
        return -1.0

    matched_total = len(shared_words) + matched_initials
    expected_total = total_parts
    score = matched_total / expected_total

    if matched_total == expected_total:
        score = min(score * 1.1, 1.0)

    return score


def _surname_focus_score(name1: str, name2: str) -> float:
    """Surname (last word) similarity via Jaro."""
    w1 = name1.split()
    w2 = name2.split()
    if not w1 or not w2:
        return 0.0
    return Jaro.normalized_similarity(w1[-1], w2[-1])


def _hard_negative_penalty(name1: str, name2: str) -> float:
    """
    Detect and penalize hard negatives.
    ONLY for non-initial cases (multi-word, no single-letter tokens).

    Returns 0.0–1.0 multiplier (1.0 = no penalty).
    """
    w1 = name1.split()
    w2 = name2.split()

    # --- Single-word checks ---
    if len(w1) < 2 or len(w2) < 2:
        # Substring trap: "Apple" in "Pineapple"
        shorter = name1 if len(name1) <= len(name2) else name2
        longer = name2 if len(name1) <= len(name2) else name1
        s_clean = shorter.replace(' ', '')
        l_clean = longer.replace(' ', '')
        if s_clean in l_clean and len(s_clean) > 0:
            ratio = len(s_clean) / len(l_clean)
            if ratio < 0.65:
                return 0.4  # heavy penalty
        return 1.0

    # --- Jr / Sr / generational suffix ---
    suffixes = {'jr', 'sr', 'i', 'ii', 'iii', 'iv', 'v', 'junior', 'senior'}
    last1, last2 = w1[-1], w2[-1]
    if last1 in suffixes or last2 in suffixes:
        if last1 != last2:
            prefix1 = ' '.join(w1[:-1])
            prefix2 = ' '.join(w2[:-1])
            if fuzz.ratio(prefix1, prefix2) > 85:
                return 0.2  # very heavy penalty

    # --- Same first name, different surname ---
    surname_sim = fuzz.ratio(w1[-1], w2[-1]) / 100.0
    firstname_sim = fuzz.ratio(w1[0], w2[0]) / 100.0

    if firstname_sim > 0.85 and surname_sim < 0.45:
        return 0.3 + 0.3 * surname_sim

    if surname_sim > 0.85 and firstname_sim < 0.45:
        return 0.35 + 0.3 * firstname_sim

    return 1.0


def _short_word_penalty(name1: str, name2: str) -> float:
    """
    Short names (<=4 chars) that differ even by 1 char are risky.
    'Iran' vs 'Iraq' → penalty.
    """
    n1 = name1.replace(' ', '').replace('.', '')
    n2 = name2.replace(' ', '').replace('.', '')
    min_len = min(len(n1), len(n2))
    if min_len <= 4:
        edit_dist = Levenshtein.distance(n1, n2)
        if edit_dist >= 1:
            return 0.55
    return 1.0


def composite_similarity(name1: str, name2: str, fe) -> float:
    """
    Enhanced composite similarity.

    Key design:
    - When initials detected → use initial_aware_score, NO hard negative penalty
    - When no initials → full penalty system active
    - Penalties are multiplicative (applied after weighted sum)
    """

    # --- Base metrics ---
    vec1 = fe.transform(name1)
    vec2 = fe.transform(name2)
    nv1 = vec1 / (np.linalg.norm(vec1) + 1e-8)
    nv2 = vec2 / (np.linalg.norm(vec2) + 1e-8)
    cosine_sim = float(np.dot(nv1, nv2))

    fuzzy_sim = fuzz.ratio(name1, name2) / 100.0
    fuzzy_partial = fuzz.partial_ratio(name1, name2) / 100.0
    fuzzy_token_sort = fuzz.token_sort_ratio(name1, name2) / 100.0
    fuzzy_token_set = fuzz.token_set_ratio(name1, name2) / 100.0

    jaro_sim = Jaro.normalized_similarity(name1, name2)

    cons1 = fe.get_consonant_skeleton(name1)
    cons2 = fe.get_consonant_skeleton(name2)
    consonant_sim = fuzz.ratio(cons1, cons2) / 100.0 if cons1 and cons2 else 0.0

    t1, t2 = set(name1.split()), set(name2.split())
    token_sim = len(t1 & t2) / len(t1 | t2) if t1 and t2 else 0.0

    # --- Detect initials ---
    initial_score = _initial_aware_score(name1, name2)
    surname_sim = _surname_focus_score(name1, name2)
    has_initials = initial_score >= 0.0

    if has_initials:
        # =====================================================
        # INITIAL PATH - trust the initial matcher, NO penalty
        # =====================================================
        raw_score = (
            0.10 * cosine_sim +
            0.05 * fuzzy_sim +
            0.10 * fuzzy_partial +
            0.05 * fuzzy_token_set +
            0.05 * jaro_sim +
            0.05 * consonant_sim +
            0.05 * token_sim +
            0.35 * initial_score +
            0.20 * surname_sim
        )
        # No penalties applied for initial cases
        final_score = raw_score

    else:
        # =====================================================
        # STANDARD PATH - full penalty system
        # =====================================================
        raw_score = (
            0.15 * cosine_sim +
            0.12 * fuzzy_sim +
            0.10 * fuzzy_partial +
            0.08 * fuzzy_token_sort +
            0.05 * fuzzy_token_set +
            0.10 * jaro_sim +
            0.10 * consonant_sim +
            0.05 * token_sim +
            0.25 * surname_sim
        )
        # Apply hard negative penalties (multiplicative)
        hn_penalty = _hard_negative_penalty(name1, name2)
        short_penalty = _short_word_penalty(name1, name2)
        final_score = raw_score * hn_penalty * short_penalty

    return min(max(final_score, 0.0), 1.0)


print("✓ Enhanced composite_similarity defined (initials protected from penalties)")



✓ Enhanced composite_similarity defined (initials protected from penalties)


In [ ]:
# =============================================================================
# CELL 8 - Search & Score Functions
# =============================================================================
def search_similar_names(query: str, top_k: int = 5, threshold: float = 0.45) -> List[Dict]:
    """Search the index and re-rank with composite similarity."""
    q_clean = normalize_and_transliterate(query)
    q_vec = feature_extractor.transform(q_clean).reshape(1, -1).astype('float32')
    faiss.normalize_L2(q_vec)

    # Pull more candidates for re-ranking
    k_search = min(top_k * 20, len(all_names))
    scores, indices = index.search(q_vec, k_search)

    results = []
    seen = set()
    for idx, fscore in zip(indices[0], scores[0]):
        if idx >= len(all_names) or idx in seen:
            continue
        seen.add(idx)
        candidate = all_names[idx]
        refined = composite_similarity(q_clean, candidate, feature_extractor)

        # Also check aliases
        alias_score = 0.0
        aliases = df.iloc[idx]["aliases_clean"]
        if aliases:
            for alias in aliases:
                if alias:
                    s = composite_similarity(q_clean, alias, feature_extractor)
                    alias_score = max(alias_score, s)

        best_score = max(refined, alias_score)

        if best_score >= threshold:
            row = df.iloc[idx]
            results.append({
                'original_name': row['name'],
                'normalized_name': candidate,
                'similarity_score': best_score,
                'name_score': refined,
                'alias_score': alias_score,
                'faiss_score': float(fscore),
                'id': row['id'],
                'aliases': row['aliases']
            })

    results.sort(key=lambda x: x['similarity_score'], reverse=True)
    return results[:top_k]


def get_score(name1: str, name2: str) -> float:
    """Drop-in replacement for model-based get_score()."""
    n1 = normalize_and_transliterate(name1)
    n2 = normalize_and_transliterate(name2)
    return composite_similarity(n1, n2, feature_extractor)


print("✓ search_similar_names() and get_score() ready")


✓ search_similar_names() and get_score() ready


In [ ]:
# =============================================================================
# CELL 9 - Quick Sanity Check
# =============================================================================
print("=" * 80)
print("QUICK TEST")
print("=" * 80)

quick_tests = [
    ("Vladimir Putin", "V. Putin"),
    ("Vladimir Putin", "Vladimire Puting"),
    ("Barack Obama", "Barac Obma"),
    ("Xi Jinping", "Xi Jin Ping"),
    ("Donald Trump", "Donald Duck"),
]

for a, b in quick_tests:
    s = get_score(a, b)
    print(f"  '{a}' vs '{b}': {s:.3f}")


QUICK TEST
  'Vladimir Putin' vs 'V. Putin': 0.826
  'Vladimir Putin' vs 'Vladimire Puting': 0.835
  'Barack Obama' vs 'Barac Obma': 0.785
  'Xi Jinping' vs 'Xi Jin Ping': 0.794
  'Donald Trump' vs 'Donald Duck': 0.239


In [ ]:
# =============================================================================
# CELL 10 - Full Comprehensive Test Suite
# =============================================================================
scenarios = {
    "1. Exact Matches": [
        ("Vladimir Putin", "Vladimir Putin"),
        ("alice wonderland", "ALICE WONDERLAND"),
    ],
    "2. Initials": [
        ("Vladimir Putin", "V. Putin"),
        ("Vladimir Putin", "V Putin"),
        ("John F. Kennedy", "J. F. Kennedy"),
        ("George W. Bush", "G. W. Bush"),
        ("J. Rowling", "J. K. Rowling"),
        ("A. Lincoln", "Abraham Lincoln"),
    ],
    "3. Typos & Misspellings": [
        ("Barack Obama", "Barac Obma"),
        ("Zelenskyy", "Zelenski"),
        ("Michael", "Micheal"),
        ("Britney Spears", "Brittany Spares"),
        ("Vladimir Putin", "Vladimire Puting"),
        ("Vladimir Putin", "VladmirPutin"),
    ],
    "4. Spacing & Punctuation": [
        ("Xi Jinping", "Xi Jin Ping"),
        ("O'Connor", "O Connor"),
        ("Coca-Cola", "CocaCola"),
        ("Cyber Security", "Cybersecurity"),
    ],
    "5. Word Order": [
        ("Angela Merkel", "Merkel Angela"),
        ("Kim Jong Un", "Un Kim Jong"),
    ],
    "6. Hard Negatives (Should be LOW < 0.50)": [
        ("Donald Trump", "Donald Duck"),
        ("George Bush", "George Washington"),
        ("Iran", "Iraq"),
        ("Apple", "Pineapple"),
        ("Robert Downey Jr", "Robert Downey Sr"),
    ],
}

print("=" * 94)
print("COMPREHENSIVE TEST SUITE - Enhanced Vector Search")
print("=" * 94)
print(f"{'Category':<30} | {'Name A':<26} | {'Name B':<26} | {'Score':>5} | Status")
print("=" * 94)

total, passed = 0, 0
for cat, pairs in scenarios.items():
    print(f"\n{cat}")
    print("-" * 94)
    for a, b in pairs:
        s = get_score(a, b)
        total += 1
        if "Hard Negatives" in cat:
            ok = s < 0.50
        else:
            ok = s >= 0.70
        if ok:
            passed += 1
        tag = "✓ PASS" if ok else "✗ FAIL"
        print(f"{'':30} | {a:<26} | {b:<26} | {s:.3f} | {tag}")

print(f"\n{'=' * 94}")
print(f"PASSED: {passed}/{total} ({100*passed/total:.0f}%)")
print("=" * 94)


COMPREHENSIVE TEST SUITE - Enhanced Vector Search
Category                       | Name A                     | Name B                     | Score | Status

1. Exact Matches
----------------------------------------------------------------------------------------------
                               | Vladimir Putin             | Vladimir Putin             | 1.000 | ✓ PASS
                               | alice wonderland           | ALICE WONDERLAND           | 1.000 | ✓ PASS

2. Initials
----------------------------------------------------------------------------------------------
                               | Vladimir Putin             | V. Putin                   | 0.826 | ✓ PASS
                               | Vladimir Putin             | V Putin                    | 0.824 | ✓ PASS
                               | John F. Kennedy            | J. F. Kennedy              | 0.794 | ✓ PASS
                               | George W. Bush             | G. W. Bush                 | 0.